In [ ]:
!pip install backtrader
!pip install deap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 5.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === ✅ Part 1: Importing Libraries
import os
import numpy as np
import pandas as pd
import glob
import math
from datetime import datetime
from collections import defaultdict
import backtrader as bt
import gc
import random
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from functools import partial
import multiprocessing

In [ ]:
'''import zipfile
import os

zip_path = "/content/price_data-20250602T011439Z-1-001.zip"
extract_dir = "/content/Data"

# Create directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Extract zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Verify extraction
files = os.listdir(extract_dir)
print(f"✅ Extracted {len(files)} files to {extract_dir}")
print("📄 Sample files:", files[:5])
'''

'import zipfile\nimport os\n\nzip_path = "/content/price_data-20250602T011439Z-1-001.zip"\nextract_dir = "/content/Data"\n\n# Create directory if it doesn\'t exist\nos.makedirs(extract_dir, exist_ok=True)\n\n# Extract zip file\nwith zipfile.ZipFile(zip_path, \'r\') as zip_ref:\n    zip_ref.extractall(extract_dir)\n\n# Verify extraction\nfiles = os.listdir(extract_dir)\nprint(f"✅ Extracted {len(files)} files to {extract_dir}")\nprint("📄 Sample files:", files[:5])\n'

In [ ]:
# === ✅ Part 2: Use Real Price Data ===
drive_path = '/content/Data/'
price_data_path = os.path.join(drive_path, 'FinTech Insider Trading Project', 'Data', 'price_data')
signals_excel_filename = '2020-2025_transactions_with_all_predictions.xlsx'
signals_file_path = os.path.join(drive_path, 'FinTech Insider Trading Project', 'Data', 'price_data', signals_excel_filename)

In [ ]:
# === ✅ Part 3: Define Correct Paths for Real Data ===
price_data_path = "/content/drive/MyDrive/FinTech Insider Trading Project/Data/price_data"
signals_file_path = "/content/drive/MyDrive/FinTech Insider Trading Project/Data/2020-2025_transactions_with_all_predictions.csv"

# Check if price data folder exists
if not os.path.exists(price_data_path):
    raise FileNotFoundError(f"❌ Price data folder not found at: {price_data_path}")

# Check if signal file exists
if not os.path.exists(signals_file_path):
    raise FileNotFoundError(f"❌ Signal Excel file not found at: {signals_file_path}")

# Strategy hyperparameters (from your specified formulas)
strategy_params_to_adjust = {
    'rho': 0.95,
    'alpha': 1.0,
    'beta': 2.0,
    'gamma': 0.5,
    'delta': 1.0,
    'theta': 0.5,
    'lambda_param': 0.5,
    'risk_tolerance': 1.0,
    'volatility_lookback': 30,
    'rebalance_freq': 100,  # in days
    'verbose_signals': True,
    'enable_logging': True,
}

print("✅ Real price data and signal file successfully loaded.")


✅ Real price data and signal file successfully loaded.


In [ ]:
# === ✅ Part 4: Parameters ===

strategy_params_to_adjust = {
    'rho': 0.95,
    'alpha': 1.0,
    'beta': 2.0,
    'gamma': 0.5,
    'delta': 1.0,
    'theta': 0.5,
    'lambda_param': 0.5,
    'risk_tolerance': 1.0,
    'volatility_lookback': 30,
    'rebalance_freq': 1,
}

In [ ]:
# === ✅ Part 5: Strategy Class ===

class ConfidencePortfolioStrategy(bt.Strategy):
    params = (
        ('rho', 0.95), ('alpha', 1.0), ('beta', 2.0), ('gamma', 0.5),
        ('delta', 1.0), ('theta', 0.5), ('lambda_param', 0.5),
        ('risk_tolerance', 1.0), ('volatility_lookback', 30),
        ('signals_df', None), ('spy_ticker', 'SPY'), ('rebalance_freq', 1),
        ('verbose_signals', True),  # Add parameter to control signal printing
        ('enable_logging', True),  # Add parameter to control all logging
    )

    def __init__(self):
        print("Starting initiation.")
        self.day_count = 0
        self.asset_names = []
        self.spy_data = None
        self.confidence_signals = defaultdict(dict)
        self.volatility_cache = defaultdict(list)
        self.last_portfolio_value = None

        if self.p.signals_df is not None:
            self._process_signals_df()

        for data in self.datas:
            name = data._name
            self.asset_names.append(name)
            if name == self.p.spy_ticker:
                self.spy_data = data

        self.log(f"\n=== Strategy Initialized ===")
        self.log(f"Assets: {len(self.asset_names)} symbols")
        self.log(f"SPY ticker: {self.p.spy_ticker}")
        self.log(f"Rebalance frequency: Every {self.p.rebalance_freq} days")
        self.log(f"Risk tolerance: {self.p.risk_tolerance}")

        # Count signals by asset
        assets_with_signals = [asset for asset in self.confidence_signals if self.confidence_signals[asset]]
        if assets_with_signals:
            self.log(f"Signals loaded for {len(assets_with_signals)} assets")
        else:
            self.log("No signals loaded")
        self.log("=" * 50)

    def log(self, message):
        """Log function that prints messages only if logging is enabled"""
        if self.p.enable_logging:
            print(message)

    def _process_signals_df(self):
        df = self.p.signals_df
        for _, row in df.iterrows():
            date = pd.to_datetime(row['Date']).date()
            stock = row['Stock']
            signal = row['Signal']
            transaction = row['Transaction']
            self.confidence_signals[stock][date] = signal

    def next(self):
        self.day_count += 1
        current_date = self.data.datetime.date(0)
        current_value = self.broker.getvalue()

        # Log daily progress
        self.log(f"\n--- Day {self.day_count}: {current_date} ---")
        daily_log = f"Portfolio Value: ${current_value:,.2f}"

        if self.last_portfolio_value is not None:
            daily_return = (current_value - self.last_portfolio_value) / self.last_portfolio_value * 100
            daily_log += f" (Daily Return: {daily_return:+.2f}%)"

        self.log(daily_log)
        self.last_portfolio_value = current_value

        # Check if it's a rebalancing day
        if self.day_count % self.p.rebalance_freq != 0:
            return

        self.log(f"\n>>> REBALANCING DAY {self.day_count} <<<")

        # Update volatilities and calculate weights
        self._update_volatilities()
        weights = self._calculate_portfolio_weights(current_date)

        # Log current positions before rebalancing (only non-zero)
        self.log("\nCurrent Positions:")
        has_positions = False
        for i, (name, data) in enumerate(zip(self.asset_names, self.datas)):
            position = self.getposition(data)
            if position.size != 0:
                position_value = position.size * data.close[0]
                self.log(f"  {name}: {position.size} shares @ ${data.close[0]:.2f} = ${position_value:,.2f}")
                has_positions = True

        if not has_positions:
            self.log("  No positions")

        # Log calculated weights (only non-zero)
        self.log("\nCalculated Weights (non-zero only):")
        non_zero_weights = {name: weight for name, weight in weights.items() if abs(weight) > 0.001}
        if non_zero_weights:
            for name, weight in non_zero_weights.items():
                self.log(f"  {name}: {weight:+.3f} ({weight*100:+.1f}%)")
        else:
            self.log("  All weights are zero")

        # Rebalance
        self._rebalance_portfolio(weights)

    def _update_volatilities(self):
        volatility_updates = []
        for i, data in enumerate(self.datas):
            name = self.asset_names[i]
            if len(data) < 2:
                continue
            daily_return = (data.close[0] - data.close[-1]) / data.close[-1]
            self.volatility_cache[name].append(daily_return)
            if len(self.volatility_cache[name]) > self.p.volatility_lookback:
                self.volatility_cache[name].pop(0)

            # Only log if we have enough data for meaningful volatility
            if len(self.volatility_cache[name]) >= 5:
                vol = self._get_volatility(name)
                volatility_updates.append((name, vol))

        # Only log if there are updates
        if volatility_updates:
            self.log("\nUpdating Volatilities:")
            for name, vol in volatility_updates:
                self.log(f"  {name}: Volatility = {vol:.3f} (annualized)")

    def _get_volatility(self, name):
        returns = self.volatility_cache[name]
        return np.std(returns) * np.sqrt(252) if len(returns) >= 5 else 0.2

    def _get_confidence_signal(self, name, date):
        signals = self.confidence_signals.get(name, {})
        recent_dates = [d for d in signals if d <= date]
        if not recent_dates:
            return 0.5
        most_recent = max(recent_dates)
        raw_signal = signals[most_recent]
        decay_factor = self.p.rho ** (date - most_recent).days
        decayed_signal = raw_signal * decay_factor

        # Only log signal details for non-default signals and recent ones
        days_old = (date - most_recent).days
        if self.p.verbose_signals and days_old <= 7 and abs(raw_signal - 0.5) > 0.001:
            self.log(f"  Signal: {name}: {raw_signal:.3f} from {most_recent} ({days_old} days old), Decayed={decayed_signal:.3f}")

        return decayed_signal

    def _calculate_portfolio_weights(self, date):
        self.log("\nCalculating Portfolio Weights:")

        # Get confidence signals
        confidences = {a: self._get_confidence_signal(a, date) for a in self.asset_names}

        # Only log non-default confidence signals
        non_default_confidences = {name: conf for name, conf in confidences.items()
                                   if abs(conf - 0.5) > 0.001}
        if non_default_confidences:
            self.log("\nActive Confidence Signals:")
            for name, conf in non_default_confidences.items():
                self.log(f"  {name}: {conf:.3f}")
        else:
            self.log("\nNo active confidence signals (all at default 0.5)")

        # Calculate strengths
        strengths = {a: abs(c - 0.5) * 2 for a, c in confidences.items()}
        tcd = sum(strengths.values())
        self.log(f"\nTotal Confidence Dispersion (TCD): {tcd:.3f}")

        if tcd == 0:
            self.log("TCD is 0, using equal weights")
            return {a: 1.0 / len(self.asset_names) for a in self.asset_names}

        # Calculate SPY preference factor
        spf = self.p.alpha * math.exp(-self.p.beta * tcd)
        self.log(f"SPY Preference Factor: {spf:.3f}")

        # Calculate base allocations
        base_alloc = {}
        for a in self.asset_names:
            cs = strengths[a]
            if a == self.p.spy_ticker:
                base_alloc[a] = spf + (1 - spf) * cs / tcd
            else:
                alloc = (1 - spf) * cs / tcd
                base_alloc[a] = -alloc if confidences[a] < 0.5 else alloc

        # Log base allocations (only non-zero)
        non_zero_base = {name: alloc for name, alloc in base_alloc.items()
                         if abs(alloc) > 0.001}
        if non_zero_base:
            self.log("\nBase Allocations (non-zero only):")
            for name, alloc in non_zero_base.items():
                self.log(f"  {name}: {alloc:+.3f}")

        # Risk adjustment
        pos_sum = sum(max(0, v) for v in base_alloc.values())
        neg_sum = sum(abs(min(0, v)) for v in base_alloc.values())
        rt = self.p.delta * self.p.risk_tolerance

        risk_adj_alloc = {}
        for a, ba in base_alloc.items():
            vol = self._get_volatility(a)
            va = 1 / (vol ** self.p.gamma) if vol > 0 else 1.0
            if ba > 0 and pos_sum > 0:
                ra = (ba / pos_sum) * va * rt
            elif ba < 0 and neg_sum > 0:
                ra = (ba / neg_sum) * va * rt * (-1)
            else:
                ra = 0
            risk_adj_alloc[a] = ra

        # Log risk-adjusted allocations (only non-zero)
        non_zero_risk_adj = {name: alloc for name, alloc in risk_adj_alloc.items()
                             if abs(alloc) > 0.001}
        if non_zero_risk_adj:
            self.log("\nRisk-Adjusted Allocations (non-zero only):")
            for name, alloc in non_zero_risk_adj.items():
                self.log(f"  {name}: {alloc:+.3f}")

        # Final normalization
        if self.p.risk_tolerance < self.p.theta:
            self.log(f"\nRisk tolerance ({self.p.risk_tolerance}) < theta ({self.p.theta}), using long-only portfolio")
            total_pos = sum(max(0, v) for v in risk_adj_alloc.values())
            final_weights = {a: max(0, v) / total_pos if total_pos else 0 for a, v in risk_adj_alloc.items()}
        else:
            self.log(f"\nRisk tolerance ({self.p.risk_tolerance}) >= theta ({self.p.theta}), allowing long/short portfolio")
            pos = sum(max(0, v) for v in risk_adj_alloc.values())
            neg = sum(abs(min(0, v)) for v in risk_adj_alloc.values())
            final_weights = {
                a: ((v / pos) * self.p.lambda_param if v > 0 else (v / neg) * (-(1 - self.p.lambda_param)) if v < 0 else 0)
                for a, v in risk_adj_alloc.items()
            }

        # Log final weights summary only if there are positions
        total_long = sum(max(0, w) for w in final_weights.values())
        total_short = sum(abs(min(0, w)) for w in final_weights.values())
        if total_long > 0 or total_short > 0:
            self.log(f"\nFinal Weights Summary:")
            if total_long > 0:
                self.log(f"  Total Long: {total_long:.3f} ({total_long*100:.1f}%)")
            if total_short > 0:
                self.log(f"  Total Short: {total_short:.3f} ({total_short*100:.1f}%)")

        return final_weights

    def _rebalance_portfolio(self, weights):
        total_value = self.broker.getvalue()
        current_positions = {a: self.getposition(d).size * d.close[0] for a, d in zip(self.asset_names, self.datas)}
        target_positions = {a: weights[a] * total_value for a in self.asset_names}

        # Check if any orders are needed
        orders_to_place = []
        for a, d in zip(self.asset_names, self.datas):
            diff = target_positions[a] - current_positions[a]
            if abs(diff) > total_value * 0.001:  # Only trade if difference > 0.1% of portfolio
                target_size = int(target_positions[a] / d.close[0]) if target_positions[a] > 0 else 0
                current_size = self.getposition(d).size

                if target_size != current_size:
                    orders_to_place.append((a, d, current_size, target_size, diff))

        # Only log if there are orders to place
        if orders_to_place:
            self.log("\nExecuting Rebalance:")
            for a, d, current_size, target_size, diff in orders_to_place:
                self.log(f"  {a}: Current={current_size} shares, Target={target_size} shares, Diff=${diff:,.2f}")
                self.order_target_size(d, target_size)
            self.log(f"  Placed {len(orders_to_place)} orders")
        else:
            self.log("\nNo rebalancing needed - positions already aligned")

    def notify_order(self, order):
        if order.status == order.Completed:
            action = 'BUY' if order.isbuy() else 'SELL'
            value = abs(order.executed.value)
            self.log(f'[ORDER] {order.data._name}: {action} {order.executed.size} @ ${order.executed.price:.2f} (${value:,.0f})')


In [ ]:
# === ✅ Part 6: Run Function ===
def run_strategy(signals_df, data_feeds, **params):
    print("Running strategy.")

    cerebro = bt.Cerebro()
    params['signals_df'] = signals_df
    cerebro.addstrategy(ConfidencePortfolioStrategy, **params)
    for name, feed in data_feeds.items():
        feed._name = name
        cerebro.adddata(feed)
    cerebro.broker.setcash(1_000_000)
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

    print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
    results = cerebro.run()
    final = cerebro.broker.getvalue()
    print('Final Portfolio Value: %.2f' % final)

    res = results[0]
    print(f"Sharpe Ratio: {res.analyzers.sharpe.get_analysis().get('sharperatio', 'N/A')}")
    print(f"Total Return: {res.analyzers.returns.get_analysis().get('rtot', 0):.2%}")
    print(f"Max Drawdown: {(res.analyzers.drawdown.get_analysis().get('max', {}).get('drawdown', 0) / 100):.2%}")
    return cerebro, res


In [ ]:
# === ✅ Part 7: Load Signals and Data Feeds ===
# Load Excel signals
try:
    signals_df = pd.read_excel(signals_file_path)
    required_cols = {'Date', 'Stock', 'Signal', 'Transaction'}
    if not required_cols.issubset(signals_df.columns):
        raise ValueError(f"Missing required columns: {required_cols - set(signals_df.columns)}")
    print("✅ Signals loaded and validated.")
except Exception as e:
    signals_df = None
    print(f"🚫 Error loading signals: {e}")

# Load price data
data_feeds = {}
if os.path.exists(price_data_path):
    parquet_files = glob.glob(os.path.join(price_data_path, '*.parquet'))
    for file_path in parquet_files:
        name = os.path.basename(file_path).replace('.parquet', '')
        try:
            df = pd.read_parquet(file_path)
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
            data = bt.feeds.PandasData(dataname=df, fromdate=df.index.min(), todate=df.index.max())
            data_feeds[name] = data
        except Exception as e:
            print(f"❌ Error loading {name}: {e}")
    print(f"✅ Loaded {len(data_feeds)} data feeds.")
else:
    print("❌ price_data_path does not exist.")


🚫 Error loading signals: Excel file format cannot be determined, you must specify an engine manually.
✅ Loaded 3021 data feeds.


In [ ]:
# === ✅ Part 8: Run Strategy ===
if signals_df is not None and data_feeds:
    cerebro, strategy_result = run_strategy(
        signals_df=signals_df,
        data_feeds=data_feeds,
        **strategy_params_to_adjust
    )
else:
    print("🚫 Cannot run strategy. Missing signals or price data.")

🚫 Cannot run strategy. Missing signals or price data.


In [ ]:
# === ✅ Part 9: Genetic Algorithm Configuration ===

import time
from datetime import datetime

# ========================================
# CONFIGURATION SECTION - Adjust these values
# ========================================

# Optimization period
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'

# Genetic Algorithm parameters
POPULATION_SIZE = 5
N_GENERATIONS = 10
CROSSOVER_PROB = 0.7
MUTATION_PROB = 0.2
TOURNAMENT_SIZE = 1

# Parameter ranges for optimization
PARAM_RANGES = {
    'rho': (0.8, 0.99),
    'alpha': (0.5, 2.0),
    'beta': (1.0, 5.0),
    'gamma': (0.1, 1.0),
    'delta': (0.5, 2.0),
    'theta': (0.1, 1.0),
    'lambda_param': (0.3, 0.7),
    'risk_tolerance': (0.5, 2.0),
    'volatility_lookback': (10, 60),
    'rebalance_freq': (1, 10)
}

# Number of parallel processes (set to 1 for debugging)
N_PROCESSES = multiprocessing.cpu_count() - 1

# Global counter for tracking evaluations
evaluation_counter = 0

# ========================================
# GENETIC ALGORITHM SETUP
# ========================================

# Create fitness and individual classes
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Define the individual creation function
def create_individual():
    individual = []
    param_values = {}

    for param, (min_val, max_val) in PARAM_RANGES.items():
        if param in ['volatility_lookback', 'rebalance_freq']:
            # Integer parameters
            value = random.randint(int(min_val), int(max_val))
            individual.append(value)
            param_values[param] = value
        else:
            # Float parameters
            value = random.uniform(min_val, max_val)
            individual.append(value)
            param_values[param] = round(value, 4)

    # Print created individual parameters
    print(f"\n📊 Created new individual with parameters:")
    for param, value in param_values.items():
        print(f"   {param}: {value}")

    return creator.Individual(individual)

# Create toolbox
toolbox = base.Toolbox()
toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Define genetic operators
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.2, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

# Custom mutation to respect parameter bounds
def bounded_mutate(individual):
    # Store original values for comparison
    original_values = individual[:]

    tools.mutGaussian(individual, mu=0, sigma=0.2, indpb=0.2)

    # Track mutations
    mutations_made = []

    # Ensure parameters stay within bounds
    for i, (param, (min_val, max_val)) in enumerate(PARAM_RANGES.items()):
        if param in ['volatility_lookback', 'rebalance_freq']:
            new_val = int(np.clip(individual[i], min_val, max_val))
            if new_val != int(original_values[i]):
                mutations_made.append(f"{param}: {int(original_values[i])} → {new_val}")
            individual[i] = new_val
        else:
            new_val = np.clip(individual[i], min_val, max_val)
            if abs(new_val - original_values[i]) > 0.001:
                mutations_made.append(f"{param}: {original_values[i]:.4f} → {new_val:.4f}")
            individual[i] = new_val

    if mutations_made:
        print(f"\n🔄 Mutations applied: {', '.join(mutations_made)}")

    return individual,

toolbox.register("mutate", bounded_mutate)

# ========================================
# FITNESS FUNCTION
# ========================================

def evaluate_strategy(individual):
    """
    Run backtest with given parameters and return Sharpe ratio as fitness
    """
    global evaluation_counter
    evaluation_counter += 1
    eval_start_time = time.time()

    print(f"\n🔍 Evaluation #{evaluation_counter} starting...")

    try:
        # Map individual values to parameter dictionary
        params = {}
        param_display = []

        for i, (param_name, _) in enumerate(PARAM_RANGES.items()):
            if param_name in ['volatility_lookback', 'rebalance_freq']:
                params[param_name] = int(individual[i])
                param_display.append(f"{param_name}={int(individual[i])}")
            else:
                params[param_name] = individual[i]
                param_display.append(f"{param_name}={individual[i]:.3f}")

        print(f"   Testing: {', '.join(param_display[:3])}...")
        print(f"            {', '.join(param_display[3:6])}...")
        print(f"            {', '.join(param_display[6:])}")

        # Add fixed parameters
        params['signals_df'] = signals_df  # Assuming this is defined in your notebook
        params['spy_ticker'] = 'SPY'
        params['verbose_signals'] = False
        params['enable_logging'] = False  # Mute logging during optimization

        # Create cerebro instance
        cerebro = bt.Cerebro()

        # Add strategy with parameters
        cerebro.addstrategy(ConfidencePortfolioStrategy, **params)

        # Set broker parameters
        cerebro.broker.setcash(1000000.0)
        cerebro.broker.setcommission(commission=0.00)

        # Add analyzers
        cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe',
                          timeframe=bt.TimeFrame.Days, compression=1,
                          factor=252, annualize=True)
        cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
        cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

        # Run backtest
        cerebro, results = run_strategy(
            signals_df=signals_df,
            data_feeds=data_feeds,
            **strategy_params_to_adjust
        )

        # Extract performance metrics
        strat = results[0]
        sharpe = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0)
        returns = strat.analyzers.returns.get_analysis()
        total_return = returns.get('rtot', 0) * 100
        drawdown = strat.analyzers.drawdown.get_analysis()
        max_dd = drawdown.get('max', {}).get('drawdown', 0)

        # Handle None or invalid Sharpe values
        if sharpe is None or np.isnan(sharpe) or np.isinf(sharpe):
            print(f"   ❌ Invalid result (Sharpe=None/NaN/Inf) - Time: {time.time() - eval_start_time:.2f}s")
            return -999,  # Heavily penalize invalid results

        print(f"   ✅ Sharpe: {sharpe:.4f}, Return: {total_return:.2f}%, MaxDD: {max_dd:.2f}% - Time: {time.time() - eval_start_time:.2f}s")

        return sharpe,

    except Exception as e:
        print(f"   ❌ Error in evaluation: {str(e)} - Time: {time.time() - eval_start_time:.2f}s")
        return -999,  # Return very negative fitness for failed runs

# Register the evaluation function
toolbox.register("evaluate", evaluate_strategy)

# ========================================
# CUSTOM STATISTICS TRACKING
# ========================================

def print_generation_stats(population, gen, start_time):
    """Print detailed statistics for the current generation"""
    fitnesses = [ind.fitness.values[0] for ind in population if ind.fitness.valid]

    if fitnesses:
        print(f"\n{'='*60}")
        print(f"📈 Generation {gen} Complete - Time Elapsed: {time.time() - start_time:.1f}s")
        print(f"{'='*60}")
        print(f"Population Stats:")
        print(f"  • Best Fitness:  {max(fitnesses):.4f}")
        print(f"  • Worst Fitness: {min(fitnesses):.4f}")
        print(f"  • Avg Fitness:   {np.mean(fitnesses):.4f}")
        print(f"  • Std Dev:       {np.std(fitnesses):.4f}")
        print(f"  • Valid Individuals: {len(fitnesses)}/{len(population)}")

        # Find best individual in this generation
        best_idx = fitnesses.index(max(fitnesses))
        best_ind = [ind for ind in population if ind.fitness.valid][best_idx]

        print(f"\nBest Individual This Generation:")
        for i, (param_name, _) in enumerate(PARAM_RANGES.items()):
            if param_name in ['volatility_lookback', 'rebalance_freq']:
                print(f"  • {param_name}: {int(best_ind[i])}")
            else:
                print(f"  • {param_name}: {best_ind[i]:.4f}")

# ========================================
# OPTIMIZATION EXECUTION
# ========================================

def run_optimization():
    """
    Run the genetic algorithm optimization
    """
    global evaluation_counter
    evaluation_counter = 0
    optimization_start_time = time.time()

    print("\n" + "🚀 " * 20)
    print("STARTING GENETIC ALGORITHM OPTIMIZATION")
    print("🚀 " * 20)
    print(f"Population Size: {POPULATION_SIZE}")
    print(f"Generations: {N_GENERATIONS}")
    print(f"Optimization Period: {START_DATE} to {END_DATE}")
    print(f"Total Parameters to Optimize: {len(PARAM_RANGES)}")
    print(f"Parallel Processes: {N_PROCESSES}")
    print("-" * 80)

    # Initialize population
    print("\n🏗️  Initializing population...")
    population = toolbox.population(n=POPULATION_SIZE)
    print(f"✅ Created {len(population)} individuals")

    # Statistics to track
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Hall of fame to keep best individuals
    hof = tools.HallOfFame(5)

    # Custom evaluation of initial population
    print(f"\n📊 Evaluating initial population...")
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit

    print_generation_stats(population, 0, optimization_start_time)

    # Run the genetic algorithm generation by generation
    print("\n🧬 Starting evolution process...")

    for gen in range(1, N_GENERATIONS + 1):
        gen_start_time = time.time()
        print(f"\n{'*'*60}")
        print(f"🔄 GENERATION {gen}/{N_GENERATIONS} STARTING")
        print(f"{'*'*60}")

        # Select the next generation individuals
        print(f"\n👥 Selecting individuals for next generation...")
        offspring = toolbox.select(population, len(population))
        offspring = list(map(toolbox.clone, offspring))

        # Apply crossover
        crossover_count = 0
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER_PROB:
                print(f"\n🔀 Applying crossover between two individuals")
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
                crossover_count += 1

        print(f"✅ Crossovers performed: {crossover_count}")

        # Apply mutation
        mutation_count = 0
        for mutant in offspring:
            if random.random() < MUTATION_PROB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                mutation_count += 1

        print(f"✅ Mutations performed: {mutation_count}")

        # Evaluate individuals with invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        print(f"\n📊 Evaluating {len(invalid_ind)} modified individuals...")

        if N_PROCESSES > 1:
            # Parallel evaluation
            pool = multiprocessing.Pool(processes=N_PROCESSES)
            fitnesses = pool.map(toolbox.evaluate, invalid_ind)
            pool.close()
            pool.join()
        else:
            # Sequential evaluation
            fitnesses = list(map(toolbox.evaluate, invalid_ind))

        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Replace population
        population[:] = offspring

        # Update hall of fame
        hof.update(population)

        # Print generation statistics
        print_generation_stats(population, gen, optimization_start_time)
        print(f"📊 Total evaluations so far: {evaluation_counter}")

    # Print final results
    total_time = time.time() - optimization_start_time
    print("\n" + "🎯 " * 25)
    print("OPTIMIZATION COMPLETE")
    print("🎯 " * 25)
    print(f"Total evaluations performed: {evaluation_counter}")

    # Best individual
    best_ind = hof[0]
    best_params = {}
    for i, (param_name, _) in enumerate(PARAM_RANGES.items()):
        if param_name in ['volatility_lookback', 'rebalance_freq']:
            best_params[param_name] = int(best_ind[i])
        else:
            best_params[param_name] = round(best_ind[i], 4)

    print(f"\n🏆 Best Sharpe Ratio Found: {best_ind.fitness.values[0]:.4f}")
    print("\n📋 Optimal Parameters:")
    for param, value in best_params.items():
        print(f"  • {param}: {value}")

    # Top 5 individuals
    print("\n🏅 Top 5 Parameter Sets:")
    for i, ind in enumerate(hof):
        print(f"\n{i+1}. Sharpe Ratio: {ind.fitness.values[0]:.4f}")
        for j, (param_name, _) in enumerate(PARAM_RANGES.items()):
            if param_name in ['volatility_lookback', 'rebalance_freq']:
                print(f"   • {param_name}: {int(ind[j])}")
            else:
                print(f"   • {param_name}: {ind[j]:.4f}")

    return best_params, None, hof  # Returning None for logbook since we're doing custom tracking

# ========================================
# RUN OPTIMIZATION
# ========================================

# Make sure you have loaded your data and signals before running this
# Example setup (adjust according to your actual data):
# symbols = ['AAPL', 'GOOGL', 'MSFT', 'SPY', ...]
# data_feeds = [...]  # Your backtrader data feeds
# signals_df = pd.read_csv('your_signals.csv')  # Your signals DataFrame


# Filter data for optimization period
print(f"\n📅 Filtering data feeds for period: {START_DATE} to {END_DATE}")
for name, data in data_feeds.items():
    df_filtered = df.loc[START_DATE:END_DATE]
    data = bt.feeds.PandasData(dataname=df_filtered)
    print(f"✅ Filtered {name}")

print("🔧 Starting optimization process...")

# Run the optimization
best_params, logbook, hall_of_fame = run_optimization()

# ========================================
# VISUALIZATION (Optional)
# ========================================

# Note: Since we're not using the standard logbook, you'll need to modify this section
# or collect statistics differently if you want to plot

# ========================================
# FINAL BACKTEST WITH BEST PARAMETERS
# ========================================

print("\n" + "🎯 " * 25)
print("RUNNING FINAL BACKTEST WITH OPTIMIZED PARAMETERS")
print("🎯 " * 25)

final_start_time = time.time()

# Run a final backtest with best parameters and logging enabled
final_cerebro = bt.Cerebro()

# Add strategy with best parameters
best_params['signals_df'] = signals_df
best_params['spy_ticker'] = 'SPY'
best_params['verbose_signals'] = False
best_params['enable_logging'] = True  # Enable logging for final run

print("\n📋 Final backtest parameters:")
for param, value in best_params.items():
    if param not in ['signals_df', 'spy_ticker', 'verbose_signals', 'enable_logging']:
        print(f"  • {param}: {value}")

final_cerebro.addstrategy(ConfidencePortfolioStrategy, **best_params)

# Set broker parameters
final_cerebro.broker.setcash(1000000.0)
final_cerebro.broker.setcommission(commission=0.00)
# Add analyzers
final_cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
final_cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
final_cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
final_cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
final_cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='timereturn')

print("\n🚀 Executing final backtest...")

# Run final backtest
final_results = final_cerebro, results = run_strategy(
    signals_df=signals_df,
    data_feeds=data_feeds,
    **strategy_params_to_adjust
)

# Print final performance metrics
print("\n" + "📊 " * 25)
print("FINAL PERFORMANCE METRICS")
print("📊 " * 25)
print(f"\n💰 Final Portfolio Value: ${final_cerebro.broker.getvalue():,.2f}")
print(f"📈 Sharpe Ratio: {final_results[0].analyzers.sharpe.get_analysis()['sharperatio']:.4f}")
print(f"📉 Max Drawdown: {final_results[0].analyzers.drawdown.get_analysis()['max']['drawdown']:.2f}%")
print(f"🔄 Total Trades: {final_results[0].analyzers.trades.get_analysis()['total']['total']}")

print(f"\n🎉 Optimization and final backtest completed successfully!")

# Extract time returns for additional analysis
time_returns = final_results[0].analyzers.timereturn.get_analysis()
dates = list(time_returns.keys())
returns = list(time_returns.values())

# Convert to pandas for easier manipulation
returns_df = pd.DataFrame({'date': dates, 'returns': returns})
returns_df['date'] = pd.to_datetime(returns_df['date'])
returns_df = returns_df.set_index('date').sort_index()

# Calculate cumulative returns
returns_df['cumulative'] = (1 + returns_df['returns']).cumprod()

# Calculate drawdown over time
returns_df['running_max'] = returns_df['cumulative'].cummax()
returns_df['drawdown'] = (returns_df['cumulative'] - returns_df['running_max']) / returns_df['running_max'] * 100

# Calculate rolling Sharpe ratio (252-day window)
rolling_window = 252
returns_df['rolling_sharpe'] = returns_df['returns'].rolling(window=rolling_window).apply(
    lambda x: np.sqrt(252) * x.mean() / x.std() if x.std() != 0 else 0
)

# Get SPY data for comparison
spy_df = data_feeds['SPY'].p.dataname
spy_df['returns'] = spy_df['close'].pct_change()
spy_df['cumulative'] = (1 + spy_df['returns']).cumprod()

# Create subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Strategy vs SPY Performance
ax1 = axes[0]
ax1.plot(returns_df.index, (returns_df['cumulative'] - 1) * 100,
         label='Strategy', color='blue', linewidth=2)
ax1.plot(spy_df.index, (spy_df['cumulative'] - 1) * 100,
         label='SPY', color='gray', linewidth=2, alpha=0.7)
ax1.set_ylabel('Cumulative Return (%)')
ax1.set_title('Strategy Performance vs SPY Benchmark', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Drawdown over time
ax2 = axes[1]
ax2.fill_between(returns_df.index, 0, returns_df['drawdown'],
                 color='red', alpha=0.3)
ax2.plot(returns_df.index, returns_df['drawdown'],
         color='darkred', linewidth=1)
ax2.set_ylabel('Drawdown (%)')
ax2.set_title('Drawdown Over Time', fontsize=14)
ax2.grid(True, alpha=0.3)

# Plot 3: Rolling Sharpe Ratio
ax3 = axes[2]
ax3.plot(returns_df.index, returns_df['rolling_sharpe'],
         color='green', linewidth=2)
ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax3.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='Sharpe = 1')
ax3.axhline(y=2, color='gray', linestyle='--', alpha=0.5, label='Sharpe = 2')
ax3.set_ylabel('Rolling Sharpe Ratio')
ax3.set_xlabel('Date')
ax3.set_title(f'Rolling Sharpe Ratio ({rolling_window}-day window)', fontsize=14)
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot the original backtrader results
print("\n📊 Generating performance plot...")
cerebro.plot(style='candlestick', volume=False)